# Assemble datasets from simulations
Combine data from simulations of different network architectures

In [1]:
import numpy as np
import pandas as pd
import os
import pickle
from tqdm import tqdm
from joblib import Parallel, delayed
import re

from stoch_sim_model import *

In [2]:
# Set parameters
sim_kind = 'agent'
reg_model = ''
runs = '-5-'
comment = "sparse-reg"

d = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/'

sim_sum_list = []
parameters_nets = []
prim_diff_bias_list = []
#sec_diff_bias_list = []
cell_series_list = []
# lineage_diff_nets = []

In [3]:
# Figure out which jobs didn't run:
d_rerun = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/raw/'
run_list = [int(re.search('sim_batch_(.*?)\.', f).group(1)) for f in os.listdir(d_rerun) if 'sim_batch' in f and comment in f and runs in f]
out = [str(x) for x in [k for k in np.arange(0, 542)] if x not in run_list]
print(len(out))
print(' '.join((out)))

0



In [4]:
# # Load data from second infections
# file_list = [f for f in os.listdir(os.path.join(d, "raw"))  if runs in f and infection_type in f and comment in f]

# for f in tqdm(file_list):
#     filepath = os.path.join(os.path.join(d, "raw"), f)
#     with open(filepath, 'rb') as filename:  
#         import_dict = pickle.load(filename)

#     parameters_nets = []
#     prim_diff_bias_list = []
#     #sec_diff_bias_list = []
#     cell_series_list = []
#     # mean_prim_diff_bias = []
#     # std_prim_diff_bias = []
#     # mean_sec_diff_bias = []
#     # std_sec_diff_bias = []
#     # mean_cell_series = []
#     # std_cell_series = []
#     # mean_lineage_diff = []
#     # std_lineage_diff = []

#     parameters = np.array(import_dict["parameters"])
#     # prim_diff_bias = np.array(import_dict["prim_diff_bias"])
#     # sec_diff_bias = np.array(import_dict["sec_diff_bias"])
#     # cell_series = np.array(import_dict["cell_time_series"])
#     # lineage_diff = np.array(import_dict["lineage_diff"])
#     sim_sum = np.array(import_dict["summary_stats"])

#     # virs = np.unique(parameters[:,[4,7,13,14]], axis = 0)
    
#     # for i, vir in enumerate(virs): # Need to fix this to stop averaging over K_EI and K_EH
#     #     index = (parameters[:,4] == vir[0])*(parameters[:,7] == vir[1])*(parameters[:,13] == vir[2])*(parameters[:,14] == vir[3])
                
#     sim_sum_list.append(np.hstack((sim_sum, parameters)))

#         # mean_prim_diff_bias.append(np.mean(prim_diff_bias[index], axis = 0))
#         # std_prim_diff_bias.append(np.std(prim_diff_bias[index], axis = 0))

#         # mean_sec_diff_bias.append(np.mean(sec_diff_bias[index], axis = 0))
#         # std_sec_diff_bias.append(np.std(sec_diff_bias[index], axis = 0))
        
#         # mean_cell_series.append(np.mean(cell_series[index], axis = 0))
#         # std_cell_series.append(np.std(cell_series[index], axis = 0))
        
#         # mean_lineage_diff.append(np.mean(lineage_diff[index], axis = 0))
#         # std_lineage_diff.append(np.std(lineage_diff[index], axis = 0))

# # Save datasets
# ### (1) Summary stats
# np.save(os.path.join(d, "raw",comment+"_summary_stats"), np.vstack(sim_sum_list))
#     # np.save(os.path.join(d, "summary_stats","std",f[:-4]), std_sim_sum)
#     # ### (2) Differentiation bias
#     # np.save(os.path.join(d, "prim_diff_bias","mean",f[:-4]), mean_prim_diff_bias)
#     # np.save(os.path.join(d, "prim_diff_bias","std",f[:-4]), std_prim_diff_bias)
#     # np.save(os.path.join(d, "sec_diff_bias","mean",f[:-4]), mean_sec_diff_bias)
#     # np.save(os.path.join(d, "sec_diff_bias","std",f[:-4]), std_sec_diff_bias)
#     # ### (3) Cell time series
#     # np.save(os.path.join(d, "cell_time_series","mean",f[:-4]), mean_cell_series)
#     # np.save(os.path.join(d, "cell_time_series","std",f[:-4]), std_cell_series)
#     # ### (4) Lineage differentiation
#     # np.save(os.path.join(d, "lineage_diff","mean",f[:-4]), mean_lineage_diff)
#     # np.save(os.path.join(d, "lineage_diff","std",f[:-4]), std_lineage_diff)

In [5]:
num_cpu = 150
file_list = [f for f in os.listdir(os.path.join(d, "raw")) if runs in f and comment in f and 'sim_batch' in f]
num_files = len(file_list)

def import_dict_func(f,d):
    
    file_path = os.path.join(os.path.join(d, "raw"), f)
    with open(file_path, 'rb') as filename:  
        import_dict = pickle.load(filename)

    parameters = np.array(import_dict["parameters"])
    sim_sum = np.array(import_dict["summary_stats"])

    out = np.hstack((parameters, sim_sum))

    return out

# create dataframe of infection response statistics
var_names = np.concatenate((param_names_for_df, stat_names_for_df))
mean_df = pd.DataFrame(np.vstack(Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d) 
                                                                                                        for file_name in file_list)), 
                       columns = [i for i in var_names]).groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).mean()

In [6]:
# Save datasets
### (1) Summary stats
mean_df.to_pickle(os.path.join(d, "raw", "stacked_data"+runs+"runs"+'-'+comment)+'.pkl')
del mean_df